# RetailMax Generator
Genera datos sintéticos realistas para 7 tablas con:
  - Distribuciones realistas (horarios pico, distribuciones normales, etc.)
  - Integridad referencial completa
  - ~5% de valores nulos en campos no críticos
  - Cobertura temporal de 12 meses (2024-01-01 a 2024-12-31)
  - 3 anomalías intencionales documentadas
  - Salida en CSV, JSON y Parquet

Anomalías documentadas:
  - A1 — Transacciones duplicadas (~0.2% de TRANS_VENTAS)
  - A2 — Fechas de devolución anteriores a la transacción origen
  - A3 — Stock físico negativo en registros de inventario puntuales

In [53]:
import random
from datetime import date, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, DateType, BooleanType
)
import numpy as np
import yaml
from pathlib import Path

## Global config

In [54]:
CONFIG_PATH = Path("config.yaml")
_cfg = yaml.safe_load(CONFIG_PATH.read_text())

In [55]:
SEED = int(_cfg["generacion"]["seed"])
random.seed(SEED)
np.random.seed(SEED)
 
OUTPUT_BASE = _cfg["generacion"]["output_base"]
DATE_START = date.fromisoformat(_cfg["fechas"]["date_start"])
DATE_END = date.fromisoformat(_cfg["fechas"]["date_end"])
DATE_RANGE = (DATE_END - DATE_START).days + 1
 
NULL_RATE = float(_cfg["generacion"]["null_rate"])
ANOMALY_RATE_TX = float(_cfg["generacion"]["anomaly_rate_tx"])

VOL = _cfg["volumenes"]
                        
PAISES = ["Colombia", "Mexico", "Chile", "Peru", "Ecuador"]
CIUDADES = {
    "Colombia": ["Bogotá", "Medellín", "Cali", "Barranquilla", "Cartagena"],
    "Mexico":   ["Ciudad de Mexico", "Guadalajara", "Monterrey", "Puebla", "Tijuana"],
    "Chile":    ["Santiago", "Valparaíso", "Concepción", "La Serena", "Antofagasta"],
    "Peru":     ["Lima", "Arequipa", "Trujillo", "Cusco", "Piura"],
    "Ecuador":  ["Quito", "Guayaquil", "Cuenca", "Santo Domingo", "Machala"],
}
ALL_CITIES = [(c, p) for p, cities in CIUDADES.items() for c in cities]

CATEGORIAS_N1 = {
    "CAT1": "Alimentos y Bebidas",
    "CAT2": "Cuidado Personal e Higiene",
    "CAT3": "Hogar y Limpieza",
    "CAT4": "Electrónica y Tecnología",
    "CAT5": "Ropa y Calzado Básico",
    "CAT6": "Bebés y Maternidad",
}
CATEGORIAS_N2 = {
    "CAT1": ["CAT1_01","CAT1_02","CAT1_03","CAT1_04","CAT1_05"],
    "CAT2": ["CAT2_01","CAT2_02","CAT2_03"],
    "CAT3": ["CAT3_01","CAT3_02","CAT3_03"],
    "CAT4": ["CAT4_01","CAT4_02","CAT4_03"],
    "CAT5": ["CAT5_01","CAT5_02"],
    "CAT6": ["CAT6_01","CAT6_02"],
}
TIPOS_TIENDA = ["Hipermercado", "Supermercado Barrio", "Tienda Conveniencia"]
CANALES_VENTA = ["Tienda Física", "Ecommerce", "Marketplace"]
TIPOS_PAGO = ["Efectivo", "Tarjeta Débito", "Tarjeta Crédito", "Transferencia", "Billetera Digital"]
MOTIVOS_DEVOLUCION = ["DEFECTO_FABRICA", "PRODUCTO_DANADO", "TALLA_INCORRECTA", "CAMBIO_OPINION", "FECHA_VENCIDA", "ERROR_DESPACHO", "OTRO"]
RANGOS_EDAD = ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
GENEROS = ["M", "F", "No binario", "Prefiero no decir"]

# Distribución estacional por mes (índice 0 = enero)
SEASONAL_INDEX = [0.75, 0.70, 0.80, 0.82, 0.85, 0.90, 0.88, 0.92, 0.95, 1.00, 1.10, 1.40]
 
# Pesos de hora del día → picos 12-14h y 18-21h
HOUR_WEIGHTS = [
    0.02, 0.01, 0.01, 0.01, 0.01, 0.02,
    0.05, 0.08, 0.10, 0.10, 0.09, 0.10,
    0.12, 0.12, 0.10, 0.09, 0.08, 0.10,
    0.12, 0.12, 0.10, 0.08, 0.05, 0.03,
]
HOUR_WEIGHTS = [w / sum(HOUR_WEIGHTS) for w in HOUR_WEIGHTS]

In [56]:
def to_python(value):
    """
    Convierte tipos numpy a tipos nativos de Python.
    Spark solo acepta int, float, bool, str, date — nunca np.float64, np.int64, etc.
    Se aplica a cualquier valor antes de meterlo en un dict de fila.
    """
    if isinstance(value, np.floating): # np.float16/32/64 → float
        return float(value)
    if isinstance(value, np.integer): # np.int8/16/32/64 → int
        return int(value)
    if isinstance(value, np.bool_): # np.bool_ → bool
        return bool(value)
    return value # str, date, None ya son nativos

def maybe_null(value, rate=NULL_RATE):
    # Retorna None con probabilidad `rate`, sino el valor original.
    return None if random.random() < rate else to_python(value)
 
def random_date(start=DATE_START, end=DATE_END):
    delta = (end - start).days
    return start + timedelta(days=random.randint(0, delta))
 
def weighted_date():
    #Genera fecha con distribución estacional.
    month = random.choices(range(1, 13), weights=SEASONAL_INDEX)[0]
    max_day = [31,29,31,30,31,30,31,31,30,31,30,31][month-1]
    day = random.randint(1, max_day)
    try:
        return date(2025, month, day)
    except ValueError:
        return date(2025, month, max_day - 1)
 
def weighted_hour():
    return random.choices(range(24), weights=HOUR_WEIGHTS)[0]
 
def normal_price(mu, sigma, lo=0.5):
    return max(lo, round(np.random.normal(mu, sigma), 2))

In [57]:
SCHEMA_PROVEEDORES = StructType([
    StructField("id_proveedor", StringType(), nullable=False),
    StructField("razon_social", StringType(), nullable=False),
    StructField("pais_origen", StringType(), nullable=False),
    StructField("tiempo_repo_dias", IntegerType(), nullable=False),
    StructField("calificacion_calidad", DoubleType(), nullable=True),   # maybe_null
    StructField("activo", BooleanType(), nullable=False),
])
 
SCHEMA_ARTICULOS = StructType([
    StructField("art_id", StringType(), nullable=False),
    StructField("cod_barra", StringType(), nullable=False),
    StructField("desc_art", StringType(), nullable=False),
    StructField("id_categ_n1", StringType(), nullable=False),
    StructField("id_categ_n2", StringType(), nullable=False),
    StructField("id_categ_n3", StringType(), nullable=False),
    StructField("id_proveedor", StringType(), nullable=False),
    StructField("precio_lista", DoubleType(), nullable=False),
    StructField("peso_kg", DoubleType(), nullable=True), # maybe_null
    StructField("unid_medida", StringType(), nullable=False),
    StructField("activo", BooleanType(), nullable=False),
    StructField("fec_alta", DateType(), nullable=False),
])
 
SCHEMA_TIENDAS = StructType([
    StructField("id_tienda", StringType(), nullable=False),
    StructField("nom_tienda", StringType(), nullable=False),
    StructField("tipo_tienda", StringType(), nullable=False),
    StructField("id_ciudad", StringType(), nullable=False),
    StructField("id_pais", StringType(), nullable=False),
    StructField("metros_cuadrados", IntegerType(), nullable=False),
    StructField("activo", BooleanType(), nullable=False),
    StructField("fec_apertura", DateType(), nullable=False),
])
 
SCHEMA_MIEMBROS = StructType([
    StructField("id_miembro", StringType(), nullable=False),
    StructField("fec_registro", DateType(), nullable=False),
    StructField("id_ciudad", StringType(), nullable=False),
    StructField("genero", StringType(), nullable=True), # maybe_null
    StructField("rango_edad", StringType(), nullable=False),
    StructField("canal_pref", StringType(), nullable=False),
    StructField("activo", BooleanType(),nullable=False),
    StructField("fec_ultima_compra", DateType(), nullable=True), # maybe_null
])
 
SCHEMA_VENTAS = StructType([
    StructField("id_trans", StringType(), nullable=False),
    StructField("id_miembro", StringType(), nullable=True), # maybe_null
    StructField("id_tienda", StringType(), nullable=False),
    StructField("art_id", StringType(), nullable=False),
    StructField("fec_trans", DateType(), nullable=False),
    StructField("hra_trans", StringType(), nullable=False),
    StructField("qty_vendida", IntegerType(), nullable=False),
    StructField("precio_unitario_venta", DoubleType(), nullable=False),
    StructField("descuento_aplicado", DoubleType(), nullable=False),
    StructField("tipo_pago", StringType(), nullable=False),
    StructField("canal_venta", StringType(), nullable=False),
    StructField("_is_anomaly", BooleanType(), nullable=False),
])
 
SCHEMA_STOCK = StructType([
    StructField("id_snapshot", StringType(), nullable=False),
    StructField("art_id", StringType(), nullable=False),
    StructField("id_tienda", StringType(), nullable=False),
    StructField("fec_snapshot", DateType(), nullable=False),
    StructField("stock_fisico", IntegerType(), nullable=False),
    StructField("stock_transito", IntegerType(), nullable=True), # maybe_null
    StructField("stock_reservado", IntegerType(), nullable=True), # maybe_null
    StructField("stock_minimo_config", IntegerType(), nullable=False),
    StructField("stock_maximo_config", IntegerType(), nullable=False),
    StructField("_is_anomaly", BooleanType(), nullable=False),
])
 
SCHEMA_DEVOLUCIONES = StructType([
    StructField("id_devolucion", StringType(), nullable=False),
    StructField("id_trans_origen", StringType(), nullable=False),
    StructField("art_id", StringType(), nullable=False),
    StructField("id_tienda", StringType(),  nullable=False),
    StructField("fec_devolucion", DateType(), nullable=False),
    StructField("qty_devuelta", IntegerType(), nullable=False),
    StructField("motivo_cod", StringType(),  nullable=False),
    StructField("canal_devolucion", StringType(), nullable=False),
    StructField("estado_devolucion", StringType(), nullable=False),
    StructField("vr_reembolso", DoubleType(), nullable=True), # maybe_null
    StructField("_is_anomaly", BooleanType(), nullable=False),
])

## 1. MSTR_PROVEEDORES  — 800 registros

In [58]:
def gen_proveedores(n=VOL["mstr_proveedores"]):
  rows = []
  paises_origen = PAISES + ["Brasil", "Argentina", "Estados Unidos", "China", "España"]
  for i in range(1, n + 1):
    tiempo_repo = int(np.clip(np.random.normal(7, 3), 1, 30))
    calidad = round(float(np.clip(np.random.normal(3.8, 0.6), 1.0, 5.0)), 1)
    rows.append({
      "id_proveedor": f"PROV{i:04d}",
      "razon_social": f"Proveedor {i} S.A.",
      "pais_origen": random.choice(paises_origen),
      "tiempo_repo_dias": tiempo_repo,
      "calificacion_calidad": maybe_null(calidad),
      "activo": random.random() > 0.05,   # 95 % activos
    })
  return rows

##  2. MSTR_ARTICULOS  — 5,000 registros

In [59]:
def gen_articulos(proveedores, n=VOL["mstr_articulos"]):
  prov_ids = [p["id_proveedor"] for p in proveedores]
  # Pesos de categoría: alimentación es la más grande
  cat_weights = [0.35, 0.15, 0.15, 0.10, 0.13, 0.12]
  unidades = ["UN", "KG", "L", "PAQ", "BOX", "M"]
  rows = []
  for i in range(1, n + 1):
    cat1_key = random.choices(list(CATEGORIAS_N1.keys()), weights=cat_weights)[0]
    cat2 = random.choice(CATEGORIAS_N2[cat1_key])
    cat3 = f"{cat2}_SUB{random.randint(1,5):02d}"

    # Precio según categoría
    price_map = {"CAT1":(5,30),"CAT2":(8,50),"CAT3":(6,40),"CAT4":(50,1200),"CAT5":(15,200),"CAT6":(10,150)}
    mu, hi = price_map[cat1_key]
    precio = normal_price(mu + (hi - mu) / 2, (hi - mu) / 4, lo=0.99)

    fec_alta = random_date(date(2000, 1, 1), date(2024, 12, 31))
    rows.append({
      "art_id": f"ART{i:05d}",
      "cod_barra": f"{random.randint(100000000000, 999999999999)}",
      "desc_art": f"Artículo {cat1_key} #{i}",
      "id_categ_n1": cat1_key,
      "id_categ_n2": cat2,
      "id_categ_n3": cat3,
      "id_proveedor": random.choice(prov_ids),
      "precio_lista": precio,
      "peso_kg": maybe_null(round(random.uniform(0.05, 15.0), 3)),
      "unid_medida": random.choice(unidades),
      "activo": random.random() > 0.08,
      "fec_alta": fec_alta,
    })
  return rows

##  3. MSTR_TIENDAS  — 150 registros

In [60]:
def gen_tiendas(n=VOL["mstr_tiendas"]):
  rows = []
  # Distribuir tiendas entre países proporcionalmente
  country_dist = {"Colombia":50,"Mexico":40,"Chile":30,"Peru":20,"Ecuador":10}
  store_list = []
  for pais, cnt in country_dist.items():
    for j in range(cnt):
      city = random.choice(CIUDADES[pais])
      store_list.append((pais, city))
  random.shuffle(store_list)

  tipo_weights = [0.15, 0.55, 0.30]  # Hiper, Super, Conveniencia
  for i, (pais, ciudad) in enumerate(store_list[:n], start=1):
    tipo = random.choices(TIPOS_TIENDA, weights=tipo_weights)[0]
    metros = {
      "Hipermercado": int(np.random.normal(8000, 1500)),
      "Supermercado Barrio": int(np.random.normal(1200, 300)),
      "Tienda Conveniencia": int(np.random.normal(250, 60)),
    }[tipo]
    metros = max(80, metros)
    fec_apertura = random_date(date(1998, 1, 1), date(2024, 6, 30))
    rows.append({
      "id_tienda":       f"TIENDA{i:03d}",
      "nom_tienda":      f"RetailMax {ciudad} {i:03d}",
      "tipo_tienda":     tipo,
      "id_ciudad":       ciudad,
      "id_pais":         pais,
      "metros_cuadrados": metros,
      "activo":          random.random() > 0.03,
      "fec_apertura":    fec_apertura,
    })
  return rows

## 4. CRM_MIEMBROS  — 50,000 registros

In [61]:
def gen_miembros(n=VOL["crm_miembros"]):
  # Distribución de edad: normal centrada en 35 años
  age_weights = [0.10, 0.25, 0.28, 0.20, 0.12, 0.05]
  canal_pref_weights = [0.50, 0.30, 0.20]
  rows = []
  for i in range(1, n + 1):
    city, pais = random.choice(ALL_CITIES)
    fec_reg = random_date(date(2010, 1, 1), date(2024, 12, 31))
    fec_ult = random_date(fec_reg, DATE_END)
    rows.append({
      "id_miembro": f"MBR{i:07d}",
      "fec_registro": fec_reg,
      "id_ciudad": city,
      "genero": maybe_null(random.choice(GENEROS)),
      "rango_edad": random.choices(RANGOS_EDAD, weights=age_weights)[0],
      "canal_pref": random.choices(CANALES_VENTA, weights=canal_pref_weights)[0],
      "activo": random.random() > 0.10,
      "fec_ultima_compra": maybe_null(fec_ult),
    })
  return rows

## 5. TRANS_VENTAS  — 1,000,000 registros  (+anomalías A1)

In [62]:
def gen_ventas(articulos, tiendas, miembros, n=VOL["trans_ventas"]):
  art_ids = [a["art_id"] for a in articulos if a["activo"]]
  art_price = {a["art_id"]: a["precio_lista"] for a in articulos}
  tienda_ids = [t["id_tienda"] for t in tiendas if t["activo"]]
  miembro_ids = [m["id_miembro"] for m in miembros]

  # Alta rotación: 20 % de artículos generan 60 % de ventas (Pareto)
  top_art = art_ids[:int(len(art_ids)*0.20)]
  rest_art = art_ids[int(len(art_ids)*0.20):]

  rows = []
  n_anomaly_tx = int(n * ANOMALY_RATE_TX)   # A1: duplicados

  for i in range(1, n + 1):
    fec = weighted_date()
    hora = weighted_hour()
    minuto = random.randint(0, 59)

    art = (random.choice(top_art) if random.random() < 0.60 else random.choice(rest_art))
    precio_base = art_price.get(art, 10.0)

    # Descuento mayor en diciembre (temporada)
    max_dscto = 0.30 if fec.month == 12 else 0.15
    descuento = round(random.uniform(0, max_dscto), 4)
    qty = max(1, int(np.random.exponential(1.5)))

    canal = random.choices(CANALES_VENTA, weights=[0.62, 0.25, 0.13])[0]
    rows.append({
      "id_trans": f"TX{i:08d}",
      "id_miembro": maybe_null(random.choice(miembro_ids), 0.08),
      "id_tienda": random.choice(tienda_ids),
      "art_id": art,
      "fec_trans": fec,
      "hra_trans": f"{hora:02d}:{minuto:02d}:00",
      "qty_vendida": qty,
      "precio_unitario_venta": float(round(precio_base * (1 - descuento), 2)),
      "descuento_aplicado": descuento,
      "tipo_pago": random.choice(TIPOS_PAGO),
      "canal_venta": canal,
      "_is_anomaly": False,
    })

  # ANOMALÍA A1: Duplicar ~0.2 % de transacciones (mismo id_trans)
  sample_idx = random.sample(range(len(rows)), n_anomaly_tx)
  duplicates = []
  for idx in sample_idx:
    dup = dict(rows[idx]) # mismo id_trans = duplicado intencional
    dup["_is_anomaly"] = True
    duplicates.append(dup)
  rows.extend(duplicates)
  random.shuffle(rows)
  return rows

## 6. INV_STOCK_DIARIO  — 750,000 registros  (+anomalía A3)

In [63]:
def gen_stock(articulos, tiendas, n=VOL["inv_stock_diario"]):
  art_ids = [a["art_id"] for a in articulos if a["activo"]]
  tienda_ids = [t["id_tienda"] for t in tiendas if t["activo"]]

  n_per_day = n // DATE_RANGE
  rows = []
  snap_id = 1

  # Alta rotación = menores stocks mínimos pero mayor movimiento
  high_rot = set(art_ids[:int(len(art_ids)*0.20)])

  for day_offset in range(DATE_RANGE):
    fec = DATE_START + timedelta(days=day_offset)
    season = SEASONAL_INDEX[fec.month - 1]

    for _ in range(n_per_day+1):
      art = random.choice(art_ids)
      is_high = art in high_rot

      stock_min = random.randint(5, 20) if is_high else random.randint(2, 10)
      stock_max = stock_min * random.randint(4, 10)
      mu_stock = (stock_min + stock_max) / 2 * season
      stock_fis = max(0, int(np.random.normal(mu_stock, mu_stock * 0.3)))
      stock_tra = maybe_null(max(0, int(np.random.normal(stock_min, 5))))
      stock_res = maybe_null(max(0, random.randint(0, stock_min)))

      is_neg_anomaly = random.random() < 0.001  # A3
      if is_neg_anomaly:
          stock_fis = -random.randint(1, 20)

      rows.append({
          "id_snapshot": f"SNP{snap_id:09d}",
          "art_id": art,
          "id_tienda": random.choice(tienda_ids),
          "fec_snapshot": fec,
          "stock_fisico": stock_fis,
          "stock_transito": stock_tra,
          "stock_reservado": stock_res,
          "stock_minimo_config": stock_min,
          "stock_maximo_config": stock_max,
          "_is_anomaly": is_neg_anomaly,
      })
      snap_id += 1
      if len(rows) >= n:
          break
    if len(rows) >= n:
        break

  return rows

## 7. POST_DEVOLUCIONES  — 50,000 registros  (+anomalía A2)

In [64]:
def gen_devoluciones(ventas, articulos, tiendas, n=VOL["post_devoluciones"]):
  art_ids    = [a["art_id"] for a in articulos if a["activo"]]
  tienda_ids = [t["id_tienda"] for t in tiendas if t["activo"]]
  art_price  = {a["art_id"]: a["precio_lista"] for a in articulos}

  # Solo tomar muestra de ventas para origen
  sample_ventas = random.sample(ventas, min(n * 3, len(ventas)))
  # Filtrar ventas no duplicadas
  seen_tx = set()
  clean_ventas = []
  for v in sample_ventas:
    if v["id_trans"] not in seen_tx:
      seen_tx.add(v["id_trans"])
      clean_ventas.append(v)

  n_anomaly_dev = int(n * 0.02)  # A2: 2 % fecha devolución < fecha transacción
  estados = ["APROBADO", "RECHAZADO", "PENDIENTE", "PROCESADO"]
  canales_dev = ["Tienda Física", "Ecommerce", "Call Center"]

  rows = []
  used = random.sample(clean_ventas, min(n, len(clean_ventas)))

  for i, v in enumerate(used, start=1):
    fec_tx = v["fec_trans"]
    art = v["art_id"]
    qty_orig = v["qty_vendida"]

    # A2: fecha devolución antes de la transacción
    is_date_anomaly = i <= n_anomaly_dev
    if is_date_anomaly:
      fec_dev = fec_tx - timedelta(days=random.randint(1, 30))
    else:
      dias_despues = int(np.random.exponential(10))
      fec_dev = fec_tx + timedelta(days=max(1, dias_despues))
      if fec_dev > DATE_END:
        fec_dev = DATE_END

    qty_dev = random.randint(1, max(1, qty_orig))
    precio  = art_price.get(art, 10.0)
    reembolso = maybe_null(round(precio * qty_dev * random.uniform(0.8, 1.0), 2))

    rows.append({
      "id_devolucion": f"DEV{i:07d}",
      "id_trans_origen": v["id_trans"],
      "art_id": art,
      "id_tienda": random.choice(tienda_ids),
      "fec_devolucion": fec_dev,
      "qty_devuelta": qty_dev,
      "motivo_cod": random.choice(MOTIVOS_DEVOLUCION),
      "canal_devolucion": random.choice(canales_dev),
      "estado_devolucion": random.choices(estados, weights=[0.60,0.10,0.15,0.15])[0],
      "vr_reembolso": reembolso,
      "_is_anomaly": is_date_anomaly,
    })

  return rows

## ESCRITURA 

In [65]:
def build_spark():
    spark = (SparkSession.builder
             .appName("RetailMax_SyntheticData")
             .config("spark.sql.shuffle.partitions", "8")
             .config("spark.driver.memory", "4g")
             .getOrCreate())
    spark.sparkContext.setLogLevel("WARN")
    return spark

In [66]:
def write_table(spark, data: list, name: str, fmt: list, schema=None):
    #Crea DataFrame y escribe en los formatos indicados
    df = spark.createDataFrame(data, schema=schema) if schema else spark.createDataFrame(data)
 
    # Eliminar columna de control de anomalías antes de exportar
    if "_is_anomaly" in df.columns:
        df_export = df.drop("_is_anomaly")
    else:
        df_export = df

    path = f"{OUTPUT_BASE}/{name}"
    df_export.coalesce(1).write.mode("overwrite").format(fmt).option("header", "true").save(path)
    print(f"    → {fmt.upper()} guardado en {path}")

    return df_export

## DETECCIÓN ANOMALÍAS

In [67]:
def detect_anomalies(spark, df_ventas, df_stock, df_devoluciones):
  """
  Detección explícita de las 3 anomalías documentadas.
  Anomalías documentadas:
    A1 — Transacciones duplicadas (~0.2% de TRANS_VENTAS)
    A2 — Fechas de devolución anteriores a la transacción origen
    A3 — Stock físico negativo en registros de inventario puntuales
  Imprime conteos y guarda reportes en CSV.
  """

  # A1: Transacciones duplicadas (mismo id_trans aparece más de una vez)
  dup_tx = df_ventas.groupBy("id_trans").count().filter(F.col("count") > 1).withColumnRenamed("count", "n_duplicados")
  n_a1 = dup_tx.count()
  print(f"A1 — Transacciones duplicadas: {n_a1:,} id_trans con >1 aparición")
  dup_tx.coalesce(1).write.mode("overwrite").format("csv").option("header","true").save(f"{OUTPUT_BASE}/anomalias/A1_duplicados")

  # A2: Devoluciones con fecha anterior a la transacción origen
  a2 = (df_devoluciones.join(
    df_ventas.select("id_trans","fec_trans"),
    df_devoluciones.id_trans_origen == df_ventas.id_trans, "left")
        .filter(F.col("fec_devolucion") < F.col("fec_trans"))
        .select("id_devolucion","id_trans_origen","fec_devolucion","fec_trans"))
  n_a2 = a2.count()
  print(f"A2 — Devoluciones con fecha inválida (antes del origen): {n_a2:,} registros")
  a2.coalesce(1).write.mode("overwrite").format("csv").option("header","true").save(f"{OUTPUT_BASE}/anomalias/A2_fechas_invalidas")

  # A3: Stock físico negativo
  a3 = df_stock.filter(F.col("stock_fisico") < 0).select("id_snapshot","art_id","id_tienda","fec_snapshot","stock_fisico")
  n_a3 = a3.count()
  print(f"A3 — Registros con stock físico negativo: {n_a3:,} registros")
  a3.coalesce(1).write.mode("overwrite").format("csv").option("header","true").save(f"{OUTPUT_BASE}/anomalias/A3_stock_negativo")

  print("Reportes de anomalías guardados en ./output/anomalias/")

## EJECUCIÓN

In [70]:
proveedores = gen_proveedores()
articulos = gen_articulos(proveedores)
tiendas = gen_tiendas()
miembros = gen_miembros()
ventas = gen_ventas(articulos, tiendas, miembros)
stock = gen_stock(articulos, tiendas)
devoluciones = gen_devoluciones(ventas, articulos, tiendas)

In [69]:
spark = build_spark()

df_prov = write_table(spark, proveedores, "MSTR_PROVEEDORES", "csv", SCHEMA_PROVEEDORES)
df_art = write_table(spark, articulos, "MSTR_ARTICULOS", "csv", SCHEMA_ARTICULOS)
df_tda = write_table(spark, tiendas, "MSTR_TIENDAS", "csv", SCHEMA_TIENDAS)
df_mbr = write_table(spark, miembros, "CRM_MIEMBROS", "parquet", SCHEMA_MIEMBROS)
df_vta = write_table(spark, ventas, "TRANS_VENTAS", "parquet", SCHEMA_VENTAS)
df_stk = write_table(spark, stock, "INV_STOCK_DIARIO", "parquet", SCHEMA_STOCK)
df_dev = write_table(spark, devoluciones, "POST_DEVOLUCIONES", "json", SCHEMA_DEVOLUCIONES)

# Detectar y reportar anomalías
detect_anomalies(spark, df_vta, df_stk, df_dev)

    → CSV guardado en ./output/MSTR_PROVEEDORES
    → CSV guardado en ./output/MSTR_ARTICULOS
    → CSV guardado en ./output/MSTR_TIENDAS


26/03/21 16:55:38 WARN TaskSetManager: Stage 103 contains a task of very large size (1536 KiB). The maximum recommended task size is 1000 KiB.


    → PARQUET guardado en ./output/CRM_MIEMBROS


26/03/21 16:55:46 WARN TaskSetManager: Stage 104 contains a task of very large size (80475 KiB). The maximum recommended task size is 1000 KiB.


    → PARQUET guardado en ./output/TRANS_VENTAS


...

A2 — Devoluciones con fecha inválida (antes del origen): 1,003 registros


26/03/21 16:56:00 WARN TaskSetManager: Stage 126 contains a task of very large size (10043 KiB). The maximum recommended task size is 1000 KiB.
26/03/21 16:56:01 WARN TaskSetManager: Stage 131 contains a task of very large size (4306 KiB). The maximum recommended task size is 1000 KiB.


A3 — Registros con stock físico negativo: 775 registros


26/03/21 16:56:02 WARN TaskSetManager: Stage 134 contains a task of very large size (34594 KiB). The maximum recommended task size is 1000 KiB.


Reportes de anomalías guardados en ./output/anomalias/
